In [3]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

pc.list_indexes()

[
    {
        "name": "help-centre-hybrid",
        "metric": "dotproduct",
        "host": "help-centre-hybrid-pfr3q8u.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 384,
        "deletion_protection": "disabled",
        "tags": null
    },
    {
        "name": "v1",
        "metric": "cosine",
        "host": "v1-pfr3q8u.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1024,
        "deletion_protection": "disabled",
        "tags": null,
        "embed": {


In [4]:
index = pc.Index("help-centre-hybrid")

index.describe_index_stats()

/Users/adambourne/GitHub/help-centre-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {'': {'vector_count': 21}},
 'total_vector_count': 21,
 'vector_type': 'dense'}

In [6]:
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder

dense_model = SentenceTransformer('all-MiniLM-L6-v2')
sparse_model = BM25Encoder().default()

sparse_model.load("../data/bm25_values.json")

def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.5):
    """
    Perform hybrid search using both dense and sparse vectors
    alpha: 0.0 = sparse (BM25) only, 1.0 = dense only, 0.5 = equal weight
    """
    if not 0 <= alpha <= 1:
        raise ValueError("Alpha must be between 0 and 1")
        
    # Get dense vector for query
    dense_query = dense_model.encode(query).tolist()
    
    # Get sparse vector for query
    sparse_query = sparse_model.encode_queries(query)
    
    hdense, hsparse = hybrid_score_norm(dense_query, sparse_query, alpha)
    
    # Perform hybrid search
    results = index.query(
        vector=hdense,
        sparse_vector=hsparse,
        top_k=top_k,
        include_metadata=True
    )
    
    return result

hybrid_search("How do I create a multi-language form?")

NameError: name 'hybrid_score_norm' is not defined